# Bitácora de Diseño: Firmware para Brazo Robótico (Taller de Proyecto 1)
**Plataforma:** EDU-CIAA-NXP (Microcontrolador LPC4337)

En este documento se registra el diseño arquitectónico del firmware para un brazo robótico de 5 grados de libertad (GDL). El objetivo del proyecto es manipular y trasladar tubos de ensayo con líquidos de forma fluida y segura.

## 1. Estrategia de Control
Debido a la criticidad de mover líquidos sin derramarlos, el control directo de las 5 articulaciones por parte de un humano resulta ineficiente. Se optó por una **estrategia híbrida**:

- **Control Articular Voluntario:** El operador gobierna la Base, el Hombro y el Codo directamente desde el joystick.
- **Control Cartesiano/Geométrico Autónomo:** La MCU asume el control del servomotor de la muñeca. A medida que el hombro y el codo cambian de posición, el procesador calcula mediante trigonometría el ángulo de contra-rotación exacto para mantener la pinza siempre a 90° (perpendicular al suelo).

> **Decisión Crítica:** Para evitar movimientos violentos al soltar el joystick (retorno por resorte), las lecturas analógicas se mapean a **velocidad** (incrementos angulares constantes) y no a **posición absoluta**.

## 2. Arquitectura de Software Multicapa
Aprovechando la alta frecuencia del procesador ARM Cortex-M4 equipado con unidad de punto flotante (FPU), el sistema prescinde del coprocesador secundario y opera bajo un **superbucle con retardos no bloqueantes y Máquina de Estados Finitos (MEF)**.

El código en C se estructura en 4 capas estrictas para garantizar portabilidad y facilitar la depuración:

### Capas del Sistema:
1. **Capa 4 - Abstracción de Hardware (HAL):** Se interactúa con el LPC4337 mediante sAPI/LPCOpen. Configura el ADC de 10-bit para el joystick y el SCTimer para las señales PWM a 50Hz.
2. **Capa 3 - Controladores (Drivers):** `joystick.c` implementa un filtro pasa-bajos y una "zona muerta" por software para ignorar el ruido eléctrico. `servos.c` almacena los límites físicos y mapea ángulos (0°-180°) a anchos de pulso en microsegundos.
3. **Capa 2 - Lógica y Algoritmos (Middleware):** Es el motor matemático. Recibe vectores de velocidad limpios, actualiza la posición e invoca el algoritmo trigonométrico de compensación para la muñeca.
4. **Capa 1 - Planificación (App):** El archivo `main.c`. Orquesta el tiempo usando `delayRead()`, invocando la adquisición, el cómputo y la actuación a diferentes frecuencias.

## 3. Demostración Estructural (Pseudocódigo en C)
A continuación se ilustra cómo se ve reflejada esta arquitectura multicapa en el despachador principal y en el driver del joystick.

In [ ]:
/* ==================================================================
   CAPA 1: main.c (Superbucle y Temporización)
================================================================== */
#include "sapi.h"           // Capa 4
#include "joystick.h"       // Capa 3
#include "cinematica.h"     // Capa 2
#include "servos_manager.h" // Capa 3

int main(void) {
    // Inicialización de hardware y drivers
    boardConfig();
    Joystick_Init();
    Servos_Init();
    
    // Retardos no bloqueantes
    delay_t timerADC, timerPWM;
    delayConfig(&timerADC, 50); // Lee mando a 20Hz
    delayConfig(&timerPWM, 20); // Refresca PWM a 50Hz

    while(1) {
        // TAREA 1: Lectura y Cálculo
        if(delayRead(&timerADC)) {
            // La capa 3 limpia el ruido y aplica zona muerta
            Vector2D input = Joystick_LeerNormalizado();
            
            // La capa 2 hace la trigonometría para la muñeca
            Cinematica_ActualizarPostura(input);
        }
        
        // TAREA 2: Actuación Física
        if(delayRead(&timerPWM)) {
            // La capa 3 traduce grados a microsegundos para la capa 4
            Servos_ActualizarFisicamente();
        }
    }
    return 0;
}

In [ ]:
/* ==================================================================
   CAPA 3: joystick.c (Filtro y Zona Muerta)
================================================================== */
#define CENTRO_IDEAL 512
#define ZONA_MUERTA_RANGO 30

Vector2D Joystick_LeerNormalizado(void) {
    Vector2D vector;
    
    // Capa 4: Lectura cruda del silicio (ADC 10-bit)
    uint16_t x_crudo = adcRead(CH1);
    
    // Lógica de Zona Muerta por software
    if(x_crudo > (CENTRO_IDEAL - ZONA_MUERTA_RANGO) && 
       x_crudo < (CENTRO_IDEAL + ZONA_MUERTA_RANGO)) {
        vector.velocidad_x = 0; // El mando está en el centro o vibrando
    } else {
        // Normalizar a porcentaje de velocidad para la cinemática
        vector.velocidad_x = MapearPorcentaje(x_crudo);
    }
    
    return vector;
}